# Thư viện và dữ liệu

In [1]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, ParameterGrid

In [1]:
df_full = pd.read_csv('../data_raw/fact_sales_full.csv', parse_dates=['order_date'])
df_full.head()

# Đặc trưng RFM

In [1]:
rfm = df_full.groupby('customer_code').agg(
    recency   = ('order_date', lambda x: (df_full.order_date.max() - x.max()).days),
    frequency = ('so_number', 'nunique'),
    monetary  = ('line_total', 'sum')
).reset_index()
rfm['churn'] = (rfm.recency > 90).astype(int)
rfm.head()

# Lưới tham số

In [1]:
param_grid = {
    'n_estimators':  [50, 100, 200],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
}
grid = list(ParameterGrid(param_grid))
print(f'Tổng tổ hợp: {len(grid)}')

In [1]:
X = rfm[['recency','frequency','monetary']].values
y = rfm['churn'].values
auc_scores = []
for params in grid:
    clf = XGBClassifier(**params, eval_metric='logloss', random_state=42, verbosity=0)
    score = cross_val_score(clf, X, y, cv=StratifiedKFold(3), scoring='roc_auc').mean()
    auc_scores.append(score)

# Xuất kết quả

In [1]:
tuning_results = pd.DataFrame(grid)
tuning_results['auc_roc'] = auc_scores
best_idx = tuning_results.auc_roc.idxmax()
best_params_xgb = tuning_results.iloc[best_idx][['n_estimators','max_depth','learning_rate']]
best_params_xgb.to_frame().to_csv('../Forecasting Product/best_params_classifier.csv')
best_params_xgb